#**Лабораторная 2. Формирование отчётов в Apache Spark**
##**Задание**

*   Сформировать отчёт с информацией о 10 наиболее популярных языках программирования по итогам года за период с 2010 по 2020 годы. Отчёт будет отражать динамику изменения популярности языков программирования и представлять собой набор таблиц "топ-10" для каждого года.
*   Получившийся отчёт сохранить в формате Apache Parquet.

Для выполнения задания вы можете использовать любую комбинацию Spark API: RDD API, Dataset API, SQL API.

##1. Устанавливаем PySpark

In [ ]:
!pip -q install pyspark

##2. Импорты

In [ ]:
import os
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window
from google.colab import files
import shutil

##3. Пути к файлам

In [ ]:
POSTS_XML = "/content/posts_sample.xml"
LANG_CSV = "/content/programming-languages.csv"
PARQUET_OUT = "/content/top_languages.parquet"

##4. Создаём SparkSession

In [ ]:
spark = (
    SparkSession.builder
    .appName("Топ языков программирования по годам")
    .master("local[*]")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(spark.version)

4.0.2


##5. Справочник языков

In [ ]:
language_source = spark.read.option("header", True).csv(LANG_CSV)
name_column = language_source.columns[0]

language_reference = (
    language_source
    .select(F.trim(F.lower(F.col(name_column))).alias("language_name"))
    .where(F.col("language_name").isNotNull())
    .where(F.col("language_name") != "")
    .dropDuplicates()
)

language_count = language_reference.count()

print("Количество языков программирования:", language_reference.count())
language_reference.show(10, truncate=False)

Количество языков программирования: 698
+-----------------------------------+
|language_name                      |
+-----------------------------------+
|ceylon                             |
|hope                               |
|m#                                 |
|metafont                           |
|mortran                            |
|pl-11                              |
|q (equational programming language)|
|bash and                           |
|objectlogo                         |
|reia                               |
+-----------------------------------+
only showing top 10 rows


##6. Читаем XML

In [ ]:
posts_df = spark.read.text(POSTS_XML)

posts_with_tags = (
    posts_df
    .where(F.col("value").contains('CreationDate="'))
    .where(F.col("value").contains('Tags="'))
)

posts_with_year_and_tags = (
    posts_with_tags
    .select(
        F.regexp_extract("value", r'CreationDate="(\d{4})', 1).cast("int").alias("year"),
        F.regexp_extract("value", r'Tags="([^"]+)"', 1).alias("tag_sequence")
    )
    .where((F.col("year") >= 2010) & (F.col("year") <= 2020))
)

count_posts = posts_with_year_and_tags.count()

print("Количество постов после фильтрации:", count_posts)
posts_with_year_and_tags.show(10, truncate=False)

Количество постов после фильтрации: 17642
+----+--------------------------------------------------------------+
|year|tag_sequence                                                  |
+----+--------------------------------------------------------------+
|2010|&lt;c++&gt;&lt;character-encoding&gt;                         |
|2010|&lt;sharepoint&gt;&lt;infopath&gt;                            |
|2010|&lt;iphone&gt;&lt;app-store&gt;&lt;in-app-purchase&gt;        |
|2010|&lt;symfony1&gt;&lt;schema&gt;&lt;doctrine&gt;&lt;fixtures&gt;|
|2010|&lt;java&gt;                                                  |
|2010|&lt;visual-studio-2010&gt;&lt;stylecop&gt;                    |
|2010|&lt;cakephp&gt;&lt;file-upload&gt;&lt;swfupload&gt;           |
|2010|&lt;git&gt;&lt;cygwin&gt;&lt;putty&gt;                        |
|2010|&lt;drupal&gt;&lt;drupal-6&gt;                                |
|2010|&lt;php&gt;&lt;wordpress&gt;&lt;memory&gt;                    |
+----+------------------------------------------

##7. Нормализуем теги

In [ ]:
normalized_tags = (
    posts_with_year_and_tags
    .withColumn("tag_sequence", F.regexp_replace("tag_sequence", "&lt;", "<"))
    .withColumn("tag_sequence", F.regexp_replace("tag_sequence", "&gt;", ">"))
    .withColumn("tag_sequence", F.regexp_replace("tag_sequence", "<", " "))
    .withColumn("tag_sequence", F.regexp_replace("tag_sequence", ">", " "))
    .withColumn("single_tag", F.explode(F.split(F.trim(F.col("tag_sequence")), r"\s+")))
    .select(
        "year",
        F.trim(F.lower(F.col("single_tag"))).alias("language_name")
    )
    .where(F.col("language_name") != "")
)

count_normalized = normalized_tags.count()

print("Количество после нормализации:", count_normalized)
normalized_tags.show(10, truncate=False)

Количество после нормализации: 52118
+----+------------------+
|year|language_name     |
+----+------------------+
|2010|c++               |
|2010|character-encoding|
|2010|sharepoint        |
|2010|infopath          |
|2010|iphone            |
|2010|app-store         |
|2010|in-app-purchase   |
|2010|symfony1          |
|2010|schema            |
|2010|doctrine          |
+----+------------------+
only showing top 10 rows


##8. Считаем упоминания

In [ ]:
normalized_tags.createOrReplaceTempView("normalized_tags_view")
language_reference.createOrReplaceTempView("language_reference_view")

language_count_by_year = spark.sql("""
    SELECT nt.year, nt.language_name, COUNT(*) AS count
    FROM normalized_tags_view nt
    INNER JOIN language_reference_view lr
    ON nt.language_name = lr.language_name
    WHERE nt.year BETWEEN 2010 AND 2020
    GROUP BY nt.year, nt.language_name
    ORDER BY nt.year, count DESC
""")

matching_tags_count = language_count_by_year.count()

print("Тегов, совпавших со справочником:", matching_tags_count)
language_count_by_year.show(10, truncate=False)

Тегов, совпавших со справочником: 438
+----+-------------+-----+
|year|language_name|count|
+----+-------------+-----+
|2010|java         |52   |
|2010|php          |46   |
|2010|javascript   |44   |
|2010|python       |26   |
|2010|objective-c  |23   |
|2010|c            |20   |
|2010|ruby         |12   |
|2010|delphi       |8    |
|2010|applescript  |3    |
|2010|r            |3    |
+----+-------------+-----+
only showing top 10 rows


##9. Топ-10 языков по годам

In [ ]:
normalized_tags.createOrReplaceTempView("normalized_tags_view")
language_reference.createOrReplaceTempView("language_reference_view")

top_10_by_year_with_rank = spark.sql("""
    WITH ranked_tags AS (
        SELECT
            nt.year,
            nt.language_name,
            COUNT(*) AS count,
            ROW_NUMBER() OVER (PARTITION BY nt.year ORDER BY COUNT(*) DESC) AS rank
        FROM normalized_tags_view nt
        INNER JOIN language_reference_view lr
        ON nt.language_name = lr.language_name
        WHERE nt.year BETWEEN 2010 AND 2020
        GROUP BY nt.year, nt.language_name
    )
    SELECT year, language_name, count, rank AS place
    FROM ranked_tags
    WHERE rank <= 10
""")

top_10_by_year_with_rank.show(20, truncate=False)

+----+-------------+-----+-----+
|year|language_name|count|place|
+----+-------------+-----+-----+
|2010|java         |52   |1    |
|2010|php          |46   |2    |
|2010|javascript   |44   |3    |
|2010|python       |26   |4    |
|2010|objective-c  |23   |5    |
|2010|c            |20   |6    |
|2010|ruby         |12   |7    |
|2010|delphi       |8    |8    |
|2010|applescript  |3    |9    |
|2010|r            |3    |10   |
|2011|php          |102  |1    |
|2011|java         |93   |2    |
|2011|javascript   |83   |3    |
|2011|python       |37   |4    |
|2011|objective-c  |34   |5    |
|2011|c            |24   |6    |
|2011|ruby         |20   |7    |
|2011|perl         |9    |8    |
|2011|delphi       |8    |9    |
|2011|bash         |7    |10   |
+----+-------------+-----+-----+
only showing top 20 rows


##10. Сохраняем результат

In [ ]:
(top_10_by_year_with_rank
    .write
    .mode("overwrite")
    .parquet(PARQUET_OUT))

print("Результат сохранён в:", PARQUET_OUT)

Результат сохранён в: /content/top_languages.parquet


##11. Восстанавливаем данные

In [ ]:
restored = spark.read.parquet(PARQUET_OUT)
print("Количество строк в restored:", restored.count())
restored.orderBy("year", "place").show(200, truncate=False)

Количество строк в restored: 100
+----+-------------+-----+-----+
|year|language_name|count|place|
+----+-------------+-----+-----+
|2010|java         |52   |1    |
|2010|php          |46   |2    |
|2010|javascript   |44   |3    |
|2010|python       |26   |4    |
|2010|objective-c  |23   |5    |
|2010|c            |20   |6    |
|2010|ruby         |12   |7    |
|2010|delphi       |8    |8    |
|2010|applescript  |3    |9    |
|2010|r            |3    |10   |
|2011|php          |102  |1    |
|2011|java         |93   |2    |
|2011|javascript   |83   |3    |
|2011|python       |37   |4    |
|2011|objective-c  |34   |5    |
|2011|c            |24   |6    |
|2011|ruby         |20   |7    |
|2011|perl         |9    |8    |
|2011|delphi       |8    |9    |
|2011|bash         |7    |10   |
|2012|php          |154  |1    |
|2012|javascript   |132  |2    |
|2012|java         |124  |3    |
|2012|python       |69   |4    |
|2012|objective-c  |45   |5    |
|2012|ruby         |27   |6    |
|2012|c   

##12. Останавливаем SparkSession


In [ ]:
spark.stop()

##13. Скачиваем zip

In [ ]:
shutil.make_archive(
    "top_languages",
    'zip',
    "/content/top_languages.parquet"
)

files.download("top_languages.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>